### Cell A — Imports & Config

In [ ]:

import os, json, math, time, random, shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# torchvision >= 0.15 recommended
import torchvision
from torchvision import transforms
from torchvision.models.video import r3d_18, R3D_18_Weights

# Optional but nice to have
from sklearn.metrics import classification_report, confusion_matrix

# ---- Paths ----
ROOT = Path("..").resolve()
DATA = ROOT / "data" / "wlasl_preprocessed"
RUNS = ROOT / "runs"
RUNS.mkdir(parents=True, exist_ok=True)

# Try to auto-pick a balanced manifest (e.g., top-N) otherwise fall back
CANDIDATES = list(DATA.glob("manifest_nslt2000_roi_*balanced*.csv")) + \
             list(DATA.glob("manifest_nslt2000_roi_final.csv")) + \
             list(DATA.glob("manifest_nslt2000_merged.csv"))
MANIFEST = CANDIDATES[0] if CANDIDATES else None  # auto-pick
# Or force a specific file:
# MANIFEST = DATA / "manifest_nslt2000_roi_top5_balanced.csv"
# Force the correct balanced manifest (24 classes, 264 samples)
MANIFEST = DATA / "manifest_nslt2000_roi_top24_balanced.csv"
assert MANIFEST.exists(), MANIFEST

#assert MANIFEST and MANIFEST.exists(), f"Balanced manifest not found. Looked at: {CANDIDATES}"

# ROI videos dir
ROI_DIR = DATA / "videos_roi"
assert ROI_DIR.exists(), "videos_roi not found. Run ROI pipeline first."

# ---- Training Hyperparams ----
SEED         = 42
NUM_FRAMES   = 32         # frames per clip
IMG_SIZE     = 112        # 112x112 for speed; ROI were 224 but 112 is good for R3D-18
BATCH_SIZE   = 8
EPOCHS       = 20
LR           = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = max(2, (os.cpu_count() or 6) // 3)
PIN_MEMORY   = True

# AMP & Early stopping
USE_AMP        = True
ES_PATIENCE    = 5
ES_MIN_DELTA   = 1e-4

# Logging / Checkpoints
STAMP   = datetime.now().strftime("%Y%m%dT%H%M%S")
RUNNAME = f"r3d18_baseline_{STAMP}"
OUTDIR  = RUNS / RUNNAME
OUTDIR.mkdir(parents=True, exist_ok=True)

print("Using manifest:", MANIFEST)
print("Run dir:", OUTDIR)

# Reproducibility

# Make PyTorch/data loading deterministic & low-thread to avoid stalls
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
# If you want to debug CUDA ops blocking, uncomment next line
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_num_threads(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Using manifest: /home/falasoul/notebooks/USD/AAI-590/Capstone/AAI-590-G3-ASL/data/wlasl_preprocessed/manifest_nslt2000_roi_top24_balanced.csv
Run dir: /home/falasoul/notebooks/USD/AAI-590/Capstone/AAI-590-G3-ASL/runs/r3d18_baseline_20251111T230525


device(type='cuda')

### Cell B — Load Manifest & Label Map

In [4]:
# === Cell B (robust) — Load manifest & build/validate class map ===
df = pd.read_csv(MANIFEST)
assert {"video_id","path","gloss","label","split"}.issubset(df.columns)
print(f"Loaded {len(df)} samples | classes(manifest)={df['gloss'].nunique()}")
print(df.head(2))

# Normalize columns
df["gloss"] = df["gloss"].astype(str)
df["split"] = df["split"].astype(str)

# ---- pick a class_index file that actually matches the manifest ----
manifest_classes = sorted(df["gloss"].unique().tolist())
n_manifest = len(manifest_classes)

# Try to find an existing class_index_* that has the same class set
cand_maps = sorted(DATA.glob("class_index_*.json"))
class_to_idx = None
chosen_map = None
for p in cand_maps:
    try:
        m = json.loads(p.read_text())
        if set(m.keys()) == set(manifest_classes):
            class_to_idx = m
            chosen_map = p
            break
    except Exception:
        pass

# Fallback: choose a “top24” map if present and matching count
if class_to_idx is None:
    for p in cand_maps:
        if "top24" in p.name:
            m = json.loads(p.read_text())
            if len(m) == n_manifest:
                # still validate set overlap (some names could differ)
                if set(m.keys()) == set(manifest_classes):
                    class_to_idx = m
                    chosen_map = p
                    break

# Final fallback: rebuild from current manifest (sorted by gloss)
if class_to_idx is None:
    class_to_idx = {g:i for i,g in enumerate(sorted(manifest_classes))}
    chosen_map = OUTDIR / f"class_index_autogen_{n_manifest}.json"
    chosen_map.write_text(json.dumps(class_to_idx, indent=2))
    print("⚠️ Built new class_to_idx and saved to:", chosen_map)
else:
    print("Loaded class_to_idx from:", chosen_map.name)

# Map labels and validate no NaNs
df["label"] = df["gloss"].map(class_to_idx)
missing_map = df["label"].isna().sum()
if missing_map:
    # Show a few unmapped glosses to debug
    bad = df[df["label"].isna()]["gloss"].value_counts().head(10).index.tolist()
    raise ValueError(f"{missing_map} samples have unmapped glosses. Example missing: {bad[:5]}")

df["label"] = df["label"].astype(int)

# ---- fix paths to ROI if needed ----
ROI_DIR = DATA / "videos_roi"
def fix_path(p):
    p = Path(p)
    if p.exists():
        return str(p)
    # fallback by video_id-based filename in videos_roi
    try:
        vid = str(int(Path(p).stem)).zfill(5)
    except Exception:
        vid = Path(p).stem  # best effort
    alt = ROI_DIR / f"{vid}.mp4"
    return str(alt) if alt.exists() else str(p)

df["path"] = df["path"].apply(fix_path)
missing_files = (~df["path"].apply(lambda x: Path(x).exists())).sum()
print("Missing files after fix_path:", missing_files)
print("Split counts:", df["split"].value_counts().to_dict())
print("Classes (from map):", len(class_to_idx))


Loaded 264 samples | classes(manifest)=24
   video_id                                               path     gloss  \
0       639  /home/falasoul/notebooks/USD/AAI-590/Capstone/...  accident   
1       624  /home/falasoul/notebooks/USD/AAI-590/Capstone/...  accident   

   label  split  exists  label_new  
0      8  train    True          0  
1      8  train    True          0  
⚠️ Built new class_to_idx and saved to: /home/falasoul/notebooks/USD/AAI-590/Capstone/AAI-590-G3-ASL/runs/r3d18_baseline_20251111T230525/class_index_autogen_24.json
Missing files after fix_path: 0
Split counts: {'train': 192, 'val': 48, 'test': 24}
Classes (from map): 24


### Cell C — Dataset (CV2 decode → T,C,H,W)

In [5]:
# %%
# Simple transforms: resize -> center-crop -> normalize
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

def sample_indices(num_frames, target=NUM_FRAMES):
    if num_frames <= 0:
        return []
    if num_frames >= target:
        # uniform sampling
        step = num_frames / float(target)
        return [int(i*step) for i in range(target)]
    else:
        # pad by repeating last frame
        idx = list(range(num_frames))
        idx += [num_frames-1]*(target - num_frames)
        return idx

def decode_clip_cv2(path, num=NUM_FRAMES, out_size=IMG_SIZE):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        return None
    frames = []
    while True:
        ok, f = cap.read()
        if not ok: break
        frames.append(f)
    cap.release()
    if not frames:
        return None
    idxs = sample_indices(len(frames), num)
    picked = [frames[i] for i in idxs]
    # Resize shortest side to out_size then center-crop square out_size
    rs = []
    for f in picked:
        h, w = f.shape[:2]
        if h < w:
            new_h = out_size
            new_w = int(w * (out_size / h))
        else:
            new_w = out_size
            new_h = int(h * (out_size / w))
        f = cv2.resize(f, (new_w, new_h), interpolation=cv2.INTER_AREA)
        # center crop
        h, w = f.shape[:2]
        y1 = max(0, (h - out_size)//2)
        x1 = max(0, (w - out_size)//2)
        f = f[y1:y1+out_size, x1:x1+out_size]
        rs.append(f)
    # BGR -> RGB, HWC -> CHW
    arr = np.stack([cv2.cvtColor(x, cv2.COLOR_BGR2RGB) for x in rs], axis=0)  # T,H,W,3
    arr = arr.transpose(3,0,1,2)  # C,T,H,W
    arr = arr.astype(np.float32)/255.0
    # normalize
    for c in range(3):
        arr[c] = (arr[c] - MEAN[c]) / STD[c]
    return torch.from_numpy(arr)  # C,T,H,W

class VideoClipDataset(Dataset):
    def __init__(self, frame_df):
        self.df = frame_df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        path = r["path"]
        y    = int(r["label"])
        x = decode_clip_cv2(path, NUM_FRAMES, IMG_SIZE)
        if x is None:
            # Return a black clip (rare fallback) + label, caller may count misses
            x = torch.zeros(3, NUM_FRAMES, IMG_SIZE, IMG_SIZE, dtype=torch.float32)
        return x, y, path


### Cell D — DataLoaders

In [6]:
# %%
# Split by 'split' column
train_df = df[df["split"]=="train"].copy()
val_df   = df[df["split"]=="val"].copy()
test_df  = df[df["split"]=="test"].copy()

print("Splits:", {k: len(v) for k,v in {
    "train": train_df, "val": val_df, "test": test_df}.items()})

train_ds = VideoClipDataset(train_df)
val_ds   = VideoClipDataset(val_df)
test_ds  = VideoClipDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


Splits: {'train': 192, 'val': 48, 'test': 24}


### Cell E — Model, Loss, Optim, Scheduler

In [8]:
# %%
num_classes = len(class_to_idx)

# Try Kinetics-400 weights for better transfer; fall back if unavailable
try:
    weights = R3D_18_Weights.KINETICS400_IMAGENET1K_V1
    model = r3d_18(weights=weights)
    print("Loaded r3d_18 with Kinetics400+IN1K weights.")
except Exception:
    model = r3d_18(weights=None)
    print("Loaded r3d_18 with random init.")

# Replace classifier
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# Optional: class weights if you saved them earlier
CLS_WTS_FILE = DATA / "class_weights_top24.json"  # <- updated
if CLS_WTS_FILE.exists():
    wt_map = json.loads(CLS_WTS_FILE.read_text())
    wts = torch.ones(num_classes, dtype=torch.float32)
    for g, idx in class_to_idx.items():
        if g in wt_map:
            wts[idx] = float(wt_map[g])
    criterion = nn.CrossEntropyLoss(weight=wts.to(device))
    print("Using class-weighted CE loss.")
else:
    criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Cosine schedule with warmup
steps_per_epoch = max(1, len(train_loader))
total_steps = EPOCHS * steps_per_epoch
warmup_steps = int(0.1 * total_steps)

def lr_lambda(step):
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    # Cosine decay
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def accuracy(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()


Loaded r3d_18 with random init.
Using class-weighted CE loss.


/tmp/ipykernel_1807542/3943527476.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


#### Cell F — Train & Validate (Early Stopping + Checkpoint)

In [9]:
# %%
best_val = float("inf")
best_path = OUTDIR / "best.pt"
es_wait = 0

def run_one_epoch(model, loader, train=True):
    model.train(train)
    total_loss = 0.0
    total_acc  = 0.0
    total_n    = 0
    for x,y,_ in tqdm(loader, disable=False, leave=False):
        x = x.to(device, non_blocking=True)   # [B, C,T,H,W]
        y = y.to(device, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out = model(x)
                loss = criterion(out, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
        else:
            with torch.no_grad():
                out = model(x)
                loss = criterion(out, y)
        total_loss += loss.item() * x.size(0)
        total_acc  += accuracy(out, y) * x.size(0)
        total_n    += x.size(0)
    return total_loss/total_n, total_acc/total_n

hist = []
for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_acc = run_one_epoch(model, train_loader, train=True)
    vl_loss, vl_acc = run_one_epoch(model,   val_loader, train=False)

    row = {"epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc,
           "val_loss": vl_loss, "val_acc": vl_acc,
           "lr": optimizer.param_groups[0]["lr"], "sec": time.time()-t0}
    hist.append(row)
    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train {tr_loss:.4f}/{tr_acc:.3f} | "
          f"val {vl_loss:.4f}/{vl_acc:.3f} | "
          f"lr {row['lr']:.2e} | {row['sec']:.1f}s")

    # Early stopping on val_loss
    improved = (best_val - vl_loss) > ES_MIN_DELTA
    if improved:
        best_val = vl_loss
        es_wait = 0
        torch.save({"model": model.state_dict(),
                    "class_to_idx": class_to_idx,
                    "config": {
                        "NUM_FRAMES": NUM_FRAMES,
                        "IMG_SIZE": IMG_SIZE,
                        "MEAN": MEAN, "STD": STD}},
                   best_path)
        print("  ✅ Saved best:", best_path.name)
    else:
        es_wait += 1
        if es_wait >= ES_PATIENCE:
            print(f"  ⛳ Early stopping (patience={ES_PATIENCE})")
            break

pd.DataFrame(hist).to_csv(OUTDIR/"history.csv", index=False)
print("History saved:", OUTDIR/"history.csv")
print("Best val_loss:", best_val)


  0%|          | 0/24 [00:00<?, ?it/s]

/tmp/ipykernel_1807542/1199194238.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


  0%|          | 0/6 [00:00<?, ?it/s]

Epoch 01/20 | train 3.3512/0.026 | val 3.3366/0.042 | lr 1.50e-04 | 45.4s
  ✅ Saved best: best.pt


  0%|          | 0/24 [00:00<?, ?it/s]

...

  0%|          | 0/6 [00:00<?, ?it/s]

Epoch 19/20 | train 1.8949/0.474 | val 3.1563/0.167 | lr 2.28e-06 | 3.2s


  0%|          | 0/24 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

Epoch 20/20 | train 1.9400/0.484 | val 3.1646/0.167 | lr 0.00e+00 | 3.1s
History saved: /home/falasoul/notebooks/USD/AAI-590/Capstone/AAI-590-G3-ASL/runs/r3d18_baseline_20251111T230525/history.csv
Best val_loss: 3.124719738960266


### Cell G — Test Set Evaluation

In [10]:
# %%
# Load best checkpoint (in case we early-stopped)
ckpt = torch.load(best_path, map_location="cpu")
model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()

all_preds, all_labels = [], []
all_paths = []
with torch.no_grad():
    for x,y,paths in tqdm(test_loader, desc="Test"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().tolist())
        all_labels.extend(y.detach().cpu().tolist())
        all_paths.extend(paths)

print("Test samples:", len(all_labels))
print(classification_report(all_labels, all_preds, zero_division=0, digits=3))
cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(class_to_idx))))
print("Confusion matrix shape:", cm.shape)
np.save(OUTDIR/"confusion_matrix.npy", cm)
print("Saved:", OUTDIR/"confusion_matrix.npy")


Test:   0%|          | 0/3 [00:00<?, ?it/s]

Test samples: 24
              precision    recall  f1-score   support

           0      0.000     0.000     0.000         1
           1      0.000     0.000     0.000         1
           2      0.000     0.000     0.000         1
           3      0.000     0.000     0.000         1
           4      0.000     0.000     0.000         1
           5      0.000     0.000     0.000         1
           6      0.000     0.000     0.000         1
           7      0.000     0.000     0.000         1
           8      0.000     0.000     0.000         1
           9      0.000     0.000     0.000         1
          10      0.000     0.000     0.000         1
          11      0.000     0.000     0.000         1
          12      0.000     0.000     0.000         1
          13      0.500     1.000     0.667         1
          14      0.000     0.000     0.000         1
          15      0.000     0.000     0.000         1
          16      0.000     0.000     0.000         1
          

### Cell H — Quick Top-k Prediction on a Random Test Clip

In [11]:
# %%
inv_idx = {v:k for k,v in class_to_idx.items()}

def predict_one(path, topk=5):
    model.eval()
    x = decode_clip_cv2(path, NUM_FRAMES, IMG_SIZE)
    if x is None:
        print("Could not read:", path); 
        return
    x = x.unsqueeze(0).to(device)  # [1,C,T,H,W]
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).squeeze(0)
        vals, idxs = torch.topk(probs, k=min(topk, probs.numel()))
    print("Top-k for:", path)
    for p, i in zip(vals.tolist(), idxs.tolist()):
        print(f"  {inv_idx[i]:20s}  {p:.3f}")

# Pick a random test sample
if len(test_df):
    p = test_df.sample(1, random_state=SEED)["path"].values[0]
    predict_one(p, topk=5)
else:
    print("No test split found.")


Top-k for: /home/falasoul/notebooks/USD/AAI-590/Capstone/AAI-590-G3-ASL/data/wlasl_preprocessed/videos_roi/13198.mp4
  trade                 0.626
  yes                   0.073
  help                  0.056
  call                  0.035
  computer              0.028


### Cell I — Export Inference Helper (for later Live/Batch use)

In [ ]:
# %%
# Write a tiny helper to load model + mapping quickly in other notebooks
helper_py = OUTDIR / "inference_helper.py"
helper_py.write_text(f"""\
import json, torch
from pathlib import Path
from torchvision.models.video import r3d_18
import torch.nn as nn

CKPT = Path(r"{str(best_path)}")
META = torch.load(CKPT, map_location="cpu")
CLASS_TO_IDX = META["class_to_idx"]
INV_IDX = {{v:k for k,v in CLASS_TO_IDX.items()}}

def load_model(device="cpu"):
    model = r3d_18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, {len(class_to_idx)})
    model.load_state_dict(META["model"])
    model.eval().to(device)
    return model, INV_IDX, META["config"]
""")
print("Wrote helper:", helper_py)
